# Bicycle Track Analysis Example

These bicycle tracks have been extracted from video footage

In [ ]:
import holoviews as hv
from hvplot import pandas
from holoviews import opts

import sys
sys.path.append("../src")

from mobiml.datasets import CopenhagenCyclists
from bokeh.io import output_notebook

output_notebook()

opts.defaults(opts.Overlay(active_tools=['wheel_zoom']))

## Load bike data

In [ ]:
bikes = CopenhagenCyclists(r"../tests/data/test_bike.pickle")
bikes.to_df()

## Plot with intersection background

In [ ]:
bg_img = hv.RGB.load_image(r"../examples/data/intersection2.png", bounds=(0,0,640,360)) 
bg_img * bikes.to_df().hvplot.scatter(
    x="x", y="y", width=900, height=500, by="traj_id", 
    alpha=0.5, hover_cols=["traj_id", "frame", "x", "y"])

## Area Aggregation

In [ ]:
import geopandas as gpd
from shapely.geometry import box
from mobiml.transforms import AreaAggregator
from mobiml.preprocessing import TrajectoryEnricher

areas = gpd.GeoDataFrame({
    "area": [1, 2, 3],
    "geometry": [
        box(0, 250, 630, 350),   # area 1: top 
        box(0, 0, 320, 250),     # area 2: left
        box(320, 0, 630, 250),   # area 3: right
    ]
})

bikes_agg = bikes.copy()
bikes_agg = TrajectoryEnricher(bikes_agg).add_features(speed=True, direction=True)
bikes_agg.running_number_added=False

area_stats = AreaAggregator(bikes_agg).aggregate(areas, freq="5min")
area_stats[["t", "area", "point_count", "point_density", "avg_speed", "avg_direction"]]

In [ ]:
selected_time_slice = area_stats[area_stats["t"]=='2021-06-09 07:10:00'].copy()
selected_time_slice

In [ ]:
selected_pts = bikes_agg.to_df().copy()
selected_pts = selected_pts[
    (selected_pts["timestamp"] >= "2021-06-09 07:10:00") &
    (selected_pts["timestamp"] <= "2021-06-09 07:15:00")
]
selected_pts

In [ ]:
bg_img = hv.RGB.load_image(r"../examples/data/intersection2.png", bounds=(0,0,640,360)) 
bg_img = bg_img * selected_time_slice.hvplot(
    c="point_count", alpha=0.4, width=900, height=500,
    colorbar=True, line_color="white", line_width=2,
    hover_cols=["area", "t", "point_count", "point_density", "avg_speed", "avg_direction"]
)
bg_img * selected_pts.hvplot.scatter(
    x="x", y="y", width=900, height=500, by="traj_id", alpha=0.5,
    hover_cols=["speed", "direction"])